# 🌙 Night Economy Taxi Agent — 5x5 Multi-Agent System (MAS)

**Module:** Artificial Intelligence (CMP-N206-0)  
**Programme:** BSc Computer Science  
**University:** University of Roehampton  
**Student Name:** Baburam Bastola 
**Student ID:** A00022220

---

## Project Overview

This notebook implements a **Multi-Agent System (MAS)** rational AI taxi fleet operating in a **5x5 grid-based** simulation of London. This version has been downsized from 10x10 to provide a more focused evaluation of coordination efficiency in a constrained space.

The system is designed around the **agent + environment + rationality** paradigm, using **A\*** search for optimal pathfinding and a centralized dispatcher for fleet coordination.

---

## Notebook Structure

| Section | Content |
|---|---|
| **1. Setup** | Imports and configuration |
| **2. Environment** | 5x5 Grid world with multi-agent rendering |
| **3. Passenger** | Passenger class with drunk logic |
| **4. Agent** | TaxiAgent class with Agent ID support |
| **5. Dispatch logic** | Centralized nearest-neighbor coordination |
| **6. Search Strategy** | A* with Manhattan distance heuristic |
| **7. Scenario 4 (5x5)** | MAS Simulation (2 Agents vs 4 Passengers) |
| **8. Evaluation** | Efficiency metrics |
| **9. Reflection** | Fairness, Sustainability, and MAS Taxonomy |
| **10. References** | IEEE formatted sources |

---
## Section 1 — Setup & Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap
from collections import deque
import heapq
import random
import os

# Reproducibility
random.seed(42)
np.random.seed(42)

os.makedirs('assets', exist_ok=True)

print('✅ All libraries loaded successfully')
print('🌙 Night Economy 5x5 MAS — Ready to initialise')

---
## Section 2 — Environment: 5x5 Grid-London World

In [ ]:
NORMAL  = 0
BLOCKED = 1
HOTSPOT = 2
HOME    = 3

AREA_LABELS = [
    (0, 1, 'Soho'), (0, 4, 'Shoreditch'), (2, 0, 'Camden'), (4, 2, 'Brixton'),
    (3, 4, 'Islington'), (4, 4, 'Hackney'), (1, 4, 'Clapham'), (3, 0, 'Richmond'),
]

class GridWorld:
    def __init__(self, blocked_cells=None):
        self.rows = 5
        self.cols = 5
        self.grid = np.zeros((self.rows, self.cols), dtype=int)
        self.hotspot_cells = [(0, 1), (0, 4), (2, 0), (4, 2)]
        for r, c in self.hotspot_cells: self.grid[r][c] = HOTSPOT
        self.home_cells = [(3, 4), (4, 4), (1, 4), (3, 0)]
        for r, c in self.home_cells: self.grid[r][c] = HOME
        if blocked_cells is None:
            self.blocked_cells = [(0, 2), (1, 1), (2, 3), (3, 2), (4, 1)]
        else: self.blocked_cells = blocked_cells
        for r, c in self.blocked_cells: self.grid[r][c] = BLOCKED

    def is_valid(self, row, col): return 0 <= row < self.rows and 0 <= col < self.cols
    def is_blocked(self, row, col): return self.grid[row][col] == BLOCKED
    def is_hotspot(self, row, col): return self.grid[row][col] == HOTSPOT
    def get_neighbours(self, row, col):
        neighbours = []
        for dr, dc, d in [(-1,0,'N'),(1,0,'S'),(0,1,'E'),(0,-1,'W')]:
            nr, nc = row + dr, col + dc
            if self.is_valid(nr, nc) and not self.is_blocked(nr, nc):
                neighbours.append((nr, nc, d))
        return neighbours

    def render(self, agents=None, passenger_pos=None, dest_pos=None, title='Grid-London 5x5', filename=None):
        cmap = ListedColormap(['#1a2a3a', '#c0392b', '#f39c12', '#27ae60'])
        fig, ax = plt.subplots(figsize=(8, 8))
        fig.patch.set_facecolor('#0c1929')
        ax.set_facecolor('#0c1929')
        ax.imshow(self.grid, cmap=cmap, vmin=0, vmax=3, alpha=0.85)
        
        agent_colors = ['#38bdf8', '#fbbf24', '#a855f7', '#f43f5e']
        if agents:
            if not isinstance(agents, list): agents = [agents]
            for i, agent in enumerate(agents):
                color = agent_colors[i % len(agent_colors)]
                path = agent.path_taken
                if path and len(path) > 1:
                    ax.plot([p[1] for p in path], [p[0] for p in path], color=color, linewidth=2.0, alpha=0.6, zorder=3, linestyle='--', marker='o', markersize=3)
                ax.plot(agent.position[1], agent.position[0], 'D', color=color, markersize=18, zorder=7)
                ax.text(agent.position[1], agent.position[0], f'T{agent.agent_id}', ha='center', va='center', color='white', fontweight='bold', fontsize=10)

        if passenger_pos:
            ax.plot(passenger_pos[1], passenger_pos[0], 'o', color='#f472b6', markersize=18, zorder=5)
            ax.text(passenger_pos[1], passenger_pos[0], 'P', ha='center', va='center', color='white', fontweight='bold')
        if dest_pos:
            ax.plot(dest_pos[1], dest_pos[0], 's', color='#34d399', markersize=18, zorder=5)
            ax.text(dest_pos[1], dest_pos[0], 'D', ha='center', va='center', color='white', fontweight='bold')
        
        for r, c, label in AREA_LABELS:
            ax.text(c, r, label, ha='center', va='center', fontsize=8, color='white', alpha=0.6)
        ax.set_title(title, color='#7dd3fc', fontsize=15, fontweight='bold')
        if filename: plt.savefig(f'assets/{filename}.png', dpi=120, bbox_inches='tight', facecolor='#0c1929')
        plt.show()

world = GridWorld()
world.render(title='Grid-London 5x5 MAS Environment')

---
## Section 3 — Passenger Class

In [ ]:
class Passenger:
    def __init__(self, location, destination, is_drunk=None):
        self.location = location
        self.destination = destination
        self.is_drunk = is_drunk if is_drunk is not None else (random.random() < 0.30)
        self.assigned = False

    def __str__(self):
        status = 'DRUNK' if self.is_drunk else 'Sober'
        return f'Passenger at {self.location} -> Dest: {self.destination} ({status})'

def spawn_passengers(world, count=3):
    passengers = []
    for _ in range(count):
        loc = random.choice(world.hotspot_cells)
        dest = random.choice(world.home_cells)
        while dest == loc: dest = random.choice(world.home_cells)
        passengers.append(Passenger(loc, dest))
    return passengers

---
## Section 4 — Agent Class: TaxiAgent

In [ ]:
class TaxiAgent:
    def __init__(self, world, start_pos=(0, 0), agent_id=1):
        self.world = world
        self.position = start_pos
        self.agent_id = agent_id
        self.score = 0
        self.has_passenger = False
        self.passenger = None
        self.trips_complete = 0
        self.total_steps = 0
        self.path_taken = [start_pos]
        self.available = True

    def pickup(self):
        self.has_passenger = True
        self.available = False
        bonus = 0
        if self.world.is_hotspot(*self.position):
            self.score += 5
            bonus = 5
        if self.passenger.is_drunk:
            self.score -= 2
            self.total_steps += 2
        return bonus

    def dropoff(self):
        self.score += 20
        self.has_passenger = False
        self.available = True
        self.trips_complete += 1
        self.passenger = None
        return 20

    def move(self, new_pos):
        self.position = new_pos
        self.total_steps += 1
        self.score -= 1
        self.path_taken.append(new_pos)

    def __str__(self):
        return f'Taxi {self.agent_id} at {self.position} | Score: {self.score}'

---
## Section 5 — Centralized Dispatch Logic

In [ ]:
def manhattan(a, b):
    return abs(a[0]-b[0]) + abs(a[1]-b[1])

def dispatch_nearest(agents, passenger):
    available_agents = [a for a in agents if a.available]
    if not available_agents: return None
    best_agent = min(available_agents, key=lambda a: manhattan(a.position, passenger.location))
    return best_agent

---
## Section 6 — Search Strategy: A*

In [ ]:
def astar(world, start, goal):
    h = lambda a, b: abs(a[0]-b[0]) + abs(a[1]-b[1])
    heap = [(h(start, goal), 0, start, [start])]
    visited = {}
    nodes = 0
    while heap:
        f, g, curr, path = heapq.heappop(heap)
        nodes += 1
        if curr == goal: return path, nodes
        if curr in visited and visited[curr] <= g: continue
        visited[curr] = g
        for nr, nc, _ in world.get_neighbours(*curr):
            new_g = g + 1
            if (nr, nc) not in visited or visited[(nr, nc)] > new_g:
                heapq.heappush(heap, (new_g + h((nr,nc), goal), new_g, (nr, nc), path+[(nr,nc)]))
    return None, nodes

---
## Section 7 — Scenario 4 (5x5): MAS Fleet Operations

In [ ]:
def run_mas_session(world, agents, passengers, verbose=True):
    total_nodes = 0
    remaining_p = passengers.copy()
    active_trips = []
    
    while remaining_p or active_trips:
        for p in remaining_p[:]:
            agent = dispatch_nearest(agents, p)
            if agent:
                agent.available = False
                agent.passenger = p
                path_p, n1 = astar(world, agent.position, p.location)
                total_nodes += n1
                active_trips.append({'agent': agent, 'p': p, 'path': path_p[1:], 'phase': 'to_pickup'})
                remaining_p.remove(p)
        
        for trip in active_trips[:]:
            agent = trip['agent']
            if trip['path']:
                agent.move(trip['path'].pop(0))
            else:
                if trip['phase'] == 'to_pickup':
                    agent.pickup()
                    path_d, n2 = astar(world, agent.position, trip['p'].destination)
                    total_nodes += n2
                    trip['path'] = path_d[1:]
                    trip['phase'] = 'to_dest'
                else:
                    agent.dropoff()
                    active_trips.remove(trip)
    
    return total_nodes

world_5x5 = GridWorld()
agents_5x5 = [TaxiAgent(world_5x5, (0,0), 1), TaxiAgent(world_5x5, (4,4), 2)]
passengers_5x5 = spawn_passengers(world_5x5, 4)

nodes_5x5 = run_mas_session(world_5x5, agents_5x5, passengers_5x5)

world_5x5.render(agents=agents_5x5, title='Scenario 4 (5x5): MAS Fleet Results', filename='Scenario_4_5x5_MAS_Fleet')

print('\n5x5 MAS Results:')
for a in agents_5x5:
    print(f'  Agent T{a.agent_id}: Score {a.score}, Steps {a.total_steps}')

## References
[1] S. Russell and P. Norvig, *Artificial Intelligence*, 2020.
[2] GLA Economics, *London's Night-Time Economy*, 2017.
[3] Hart et al., *A formal basis for heuristic determination of minimum cost paths*, 1968.
[4] K. Lum and W. Isaac, *To predict and serve?*, 2016.
[5] TfL, *Zero Emission Capable Taxis*, 2023.